# Titanic
## Score: .78229

In [33]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
DATA = ROOT / "titanic"
assert (DATA / "train.csv").exists()

In [34]:
def process_age(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    guess_ages = np.zeros((2, 3))
    overall_med = data["Age"].median()
    for i in range(0, 2):
        for j in range(0, 3):
            guess_df = data[(data["Sex"] == i) & (data["Pclass"] == j + 1)]["Age"].dropna()
            age_guess = guess_df.median()
            if pd.isna(age_guess):
                age_guess = overall_med
            guess_ages[i, j] = int(age_guess / 0.5 + 0.5) * 0.5
    for i in range(0, 2):
        for j in range(0, 3):
            data.loc[
                (data.Age.isnull()) & (data.Sex == i) & (data.Pclass == j + 1),
                "Age",
            ] = guess_ages[i, j]
    data["Age"] = data["Age"].fillna(overall_med)
    data["Age"] = data["Age"].astype(int)
    data.loc[data["Age"] <= 16, "Age"] = 0
    data.loc[(data["Age"] > 16) & (data["Age"] <= 32), "Age"] = 1
    data.loc[(data["Age"] > 32) & (data["Age"] <= 48), "Age"] = 2
    data.loc[(data["Age"] > 48) & (data["Age"] <= 64), "Age"] = 3
    data.loc[data["Age"] > 64, "Age"] = 4
    return data


def preprocessing(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data = data.drop(columns=["Name", "PassengerId", "Ticket", "Cabin"], errors="ignore")
    tsex = {"male": 0, "female": 1}
    tembarked = {"S": 0, "C": 1, "Q": 2}
    data["Sex"] = data["Sex"].map(tsex)
    data["Embarked"] = data["Embarked"].map(tembarked)
    data = process_age(data)
    for col in data.columns:
        m = data[col].abs().max()
        if m and not pd.isna(m):
            data[col] = data[col] / m
    data["Fare"] = data["Fare"] * 1.8
    data = data.fillna(0)
    if "Survived" in data.columns:
        data = data.drop(columns="Survived")
    return data

In [35]:
train_data = pd.read_csv(DATA / "train.csv")
test_data = pd.read_csv(DATA / "test.csv")

train_data["Fare"] = train_data["Fare"].fillna(train_data["Fare"].median())
test_data["Fare"] = test_data["Fare"].fillna(train_data["Fare"].median())

train_label = train_data["Survived"]
processed_train = preprocessing(train_data)

X_train, X_val, y_train, y_val = train_test_split(
    processed_train, train_label, test_size=0.1, random_state=42
)

clf = CatBoostClassifier(
    bagging_temperature=0.2,
    metric_period=1000,
    learning_rate=0.0001,
    verbose=False,
    random_seed=42,
)
clf.fit(X_train, y_train, eval_set=(X_val, y_val), verbose=False)

prediction = clf.predict_proba(X_val)
rounded_predictions = np.argmax(prediction, axis=-1)
c_matrix = confusion_matrix(y_val, rounded_predictions)
dt_acc = c_matrix.trace() / c_matrix.sum()
print(c_matrix)
print(dt_acc)

processed_test = preprocessing(test_data)
final_predictions = clf.predict(processed_test)

output = pd.DataFrame(
    {"PassengerId": test_data["PassengerId"], "Survived": final_predictions.astype(int)}
)
out_path = ROOT / "submission.csv"
output.to_csv(out_path, index=False)
print(out_path, len(output))
output.head(10)

[[46  8]
 [ 7 29]]
0.8333333333333334
c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv 418


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
